# LSTM Text Generation

**Question:** Implement an LSTM-based text generation model and train the model to generate text sequences.

---

## What is Text Generation?
The model learns the patterns in a piece of text, then uses that knowledge to **continue writing** from a given starting point.

Example: Give it `"hello world"` and it produces `"hello world, this is a simple text generation..."`

This is **character-level** generation — the model predicts one character at a time (not whole words).

---

## LSTM vs Vanilla RNN

In notebook 11 we used a plain RNN. The problem with plain RNNs is the **vanishing gradient** — when sequences get long, gradients shrink to near-zero during backpropagation, so the model forgets what it saw at the start of the sequence.

**LSTM (Long Short-Term Memory)** solves this by having **two** memory channels instead of one:

| State | Name | Purpose |
|-------|------|---------|
| `h` | Hidden state | Short-term memory — what just happened |
| `c` | Cell state | Long-term memory — important context carried over many steps |

The LSTM uses **gates** (learned switches) to decide:
- What to **forget** from long-term memory
- What **new information** to write into long-term memory
- What to **output** from memory at each step

This lets LSTMs remember patterns across much longer sequences than plain RNNs.

---

## How it works (overview)
```
Training:  'hello worl' -> predict -> 'd'
           'ello world' -> predict -> ','
           'llo world,' -> predict -> ' '
           ... (sliding window of 10 chars)

Generating: start with seed 'hello world'
            predict next char -> append -> slide window -> repeat
```

In [ ]:
import torch
import torch.nn as nn

## Step 1 - Build the Vocabulary and Training Data

### Character vocabulary
Unlike notebook 11 (which used words), here the vocabulary is every **unique character** in the text — letters, spaces, punctuation, uppercase, etc.

We build two lookup tables:
- **`c2i`** (char-to-index): `'a'` → `3`, `'b'` → `4`, ...
- **`i2c`** (index-to-char): `3` → `'a'`, `4` → `'b'`, ...

### Sliding window (creating training pairs)
We slide a window of **10 characters** across the text. Each window is one training example:

```
Text:  h e l l o   w o r l d ,   t h i s ...

Window 1:  'hello worl'  ->  predict  'd'
Window 2:  'ello world'  ->  predict  ','
Window 3:  'llo world,'  ->  predict  ' '
Window 4:  'lo world, '  ->  predict  't'
...
```

Each character is converted to its index before being fed to the model.

In [ ]:
# The text the model will learn from
text = 'hello world, this is a simple text generation using LSTMs.'

# Get every unique character and sort them (gives a consistent ordering)
chars = sorted(set(text))
print(f'Vocabulary ({len(chars)} chars):', chars)

# Build char <-> index lookup tables
c2i = {c: i for i, c in enumerate(chars)}  # char -> number
i2c = {i: c for i, c in enumerate(chars)}  # number -> char

# Build training pairs using a sliding window of 10 characters
SEQ_LEN = 10
X, y = [], []
for i in range(len(text) - SEQ_LEN):
    window = text[i : i + SEQ_LEN]          # 10 input characters
    target = text[i + SEQ_LEN]              # 1 character to predict
    X.append([c2i[c] for c in window])      # convert chars to indices
    y.append(c2i[target])

X_train = torch.tensor(X)   # Shape: (num_windows, 10)
y_train = torch.tensor(y)   # Shape: (num_windows,)

print(f'Training examples: {len(X_train)}')
print(f'Example: "{text[:SEQ_LEN]}" -> "{text[SEQ_LEN]}"')

## Step 2 - Define the LSTM Model

The model has three layers, same structure as the RNN in notebook 11, but the middle layer is now an LSTM:

1. **`nn.Embedding(vocab_size, 16)`** — converts each character index to a 16-dimensional vector

2. **`nn.LSTM(16, 128, batch_first=True)`** — the LSTM layer
   - Input size = 16 (embedding dimension)
   - Hidden size = 128 (size of both `h` and `c` states)
   - Returns **3 things**: `(output, (h, c))`
     - `output`: hidden state at every time step — we don't need this
     - `h`: final hidden state (short-term memory) — **this is what we use**
     - `c`: final cell state (long-term memory) — we don't need this for prediction

3. **`nn.Linear(128, vocab_size)`** — maps the final hidden state to a score for every character

### Why `_, (h, _)`?
```python
_, (h, _) = self.lstm(embedded)
#^       ^--- discard cell state c
#|--- discard all intermediate outputs
```
We only care about `h` — the summary of the entire sequence.

In [ ]:
class TextLSTM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # Convert character indices to 16-dimensional vectors
        self.embed = nn.Embedding(vocab_size, 16)

        # LSTM: reads the 10-char sequence, maintains hidden + cell state
        # hidden_size=128 means both h and c are 128-dimensional vectors
        self.lstm = nn.LSTM(input_size=16, hidden_size=128, batch_first=True)

        # Map the final hidden state to a score for every character in vocabulary
        self.fc = nn.Linear(128, vocab_size)

    def forward(self, x):
        embedded = self.embed(x)             # (batch, 10) -> (batch, 10, 16)

        # LSTM returns: output (all steps), and tuple (h, c) for final states
        # We only need h (hidden state), so we discard output and c with _
        _, (h, _) = self.lstm(embedded)      # h shape: (1, batch, 128)

        return self.fc(h[-1])                # h[-1] shape: (batch, 128) -> (batch, vocab_size)

## Step 3 - Train the Model

We train for 50 epochs. Because the dataset is small (only ~48 examples from our short text), we feed all examples at once each epoch.

**CrossEntropyLoss** works here just like in the word prediction notebook — it treats next-character prediction as a classification problem with one class per unique character.

Note: The original code didn't print the loss during training. We add that here so you can see the model learning.

In [ ]:
model     = TextLSTM(len(chars))
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()  # Standard loss for classification (predicting which char comes next)

for epoch in range(50):
    model.train()
    optimizer.zero_grad()

    outputs = model(X_train)              # Forward pass: get character scores for every window
    loss   = criterion(outputs, y_train)  # Compare to true next characters

    loss.backward()   # Backpropagation through time — gradients flow back through the LSTM sequence
    optimizer.step()  # Update weights

    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}/50 | Loss: {loss.item():.4f}')

## Step 4 - Generate Text

The `generate` function extends a seed string one character at a time:

```
Step 1: seed = 'hello world'
        take last 10 chars -> 'hello worl'
        model predicts     -> 'd'
        append             -> 'hello world'

Step 2: take last 10 chars -> 'ello world'
        model predicts     -> ','
        append             -> 'hello world,'

Step 3: take last 10 chars -> 'llo world,'
        model predicts     -> ' '
        append             -> 'hello world, '
... and so on for `length` steps
```

The key idea is `curr_seq[-10:]` — we always take the **most recent 10 characters** as context, sliding the window forward with every new character we generate.

In [ ]:
def generate(seed, length=50):
    model.eval()  # Disable any dropout/batchnorm training behaviour

    # Convert the seed string into a list of character indices
    curr_seq = [c2i[c] for c in seed]
    result   = seed  # We will build the output by appending to this

    for _ in range(length):
        # Always use the last 10 characters as input (sliding window)
        window = curr_seq[-SEQ_LEN:]

        with torch.no_grad():  # No gradients needed during generation
            scores    = model(torch.tensor([window]))  # [window] adds the batch dimension
            next_idx  = scores.argmax(dim=1).item()    # Pick the highest-scoring character
            next_char = i2c[next_idx]

        result   += next_char    # Append predicted character to the output string
        curr_seq.append(next_idx)  # Append index to sequence so it becomes part of next window

    return result

# Generate 50 characters starting from the seed
print(generate('hello world', length=50))